# MobileRec: AutoRec and MultiVAE Autoencoder Models

AutoRec and MultiVAE use the same users, apps, training matrix, and chronological split as the baselines. AutoRec reconstructs a user’s interaction history from a compact hidden representation, while MultiVAE learns a probabilistic latent representation. These architectures allow a comparison of two autoencoder approaches using the same implicit interactions.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import copy
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import coo_matrix, load_npz
from torch.utils.data import DataLoader, Dataset

In [2]:
run_name = "all_users"
seed = 42
top_k = 10
batch_size = 128
max_epochs = 20
patience = 3
min_improvement = 0.00001

Training is limited to 20 epochs. Validation NDCG@10 is monitored after each epoch, and early stopping is triggered after three consecutive epochs without an improvement exceeding `0.00001`. The selected checkpoint is determined by validation performance rather than by the final training epoch.

In [3]:
split_dir = Path("artifacts") / "split_all_users"
base_dir = Path("artifacts") / "baselines_all_users"
results_dir = Path("artifacts") / "autoencoders_all_users"

train_path = split_dir / "train_all_users.parquet"
valid_path = split_dir / "validation_all_users.parquet"
test_path = split_dir / "test_all_users.parquet"
user_map_path = split_dir / "users_map_all_users.parquet"
item_map_path = split_dir / "items_map_all_users.parquet"
matrix_path = base_dir / f"user_item_matrix_{run_name}_seed_{seed}.npz"
baseline_path = base_dir / f"baseline_results_{run_name}_seed_{seed}.csv"

results_dir.mkdir(parents=True, exist_ok=True)

for path in [train_path, valid_path, test_path, user_map_path, item_map_path, matrix_path, baseline_path]:
    if not path.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

In [4]:
user_map = pd.read_parquet(user_map_path).sort_values("user_idx")
item_map = pd.read_parquet(item_map_path).sort_values("item_idx")
user_item_matrix = load_npz(matrix_path).tocsr()

num_users, num_items = user_item_matrix.shape

if num_users != len(user_map) or num_items != len(item_map):
    raise ValueError("Matrica i mapiranja se ne poklapaju.")

print("Users:", num_users)
print("Applications:", num_items)
print("Training interactions:", user_item_matrix.nnz)

Users: 700111
Applications: 10172
Training interactions: 17748311


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

set_seed(seed)

print("Uredjaj:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Uredjaj: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


The user and app mappings, chronological training, validation, and test sets, and sparse matrix from the preceding notebooks are loaded. The matrix contains 700,111 rows, 10,172 columns, and 17,748,311 unique training pairs. Its dimensions are checked against the mappings.

The models are trained on the GPU. The random seed is reset to `seed = 42` to support reproducibility of weight initialization, batch ordering, and other random components.

In [6]:
def load_targets(path):
    data = pd.read_parquet(path, columns=["uid", "app_package"])
    data = data.merge(user_map, on="uid", how="left")
    data = data.merge(item_map, on="app_package", how="left")
    targets = data.dropna(subset=["item_idx"]).copy()
    targets["user_idx"] = targets["user_idx"].astype(int)
    targets["item_idx"] = targets["item_idx"].astype(int)
    return data, targets.sort_values("user_idx").reset_index(drop=True)

valid_data, valid_targets = load_targets(valid_path)
test_data, test_targets = load_targets(test_path)

valid_coverage = len(valid_targets) / len(valid_data) * 100
test_coverage = len(test_targets) / len(test_data) * 100

print("Validation targets:", len(valid_targets))
print("Validation coverage:", round(valid_coverage, 4), "%")
print("Test targets:", len(test_targets))
print("Test coverage:", round(test_coverage, 4), "%")

Validation targets: 700108
Validation coverage: 99.9996 %
Test targets: 700096
Test coverage: 99.9979 %


The validation and test sets each contain one target per user. Text identifiers are mapped to the numeric indices used by the interaction matrix and models.

The candidate set contains only apps observed in training. Three validation and 15 test targets refer to an item cold-start app and are excluded from ranking metrics. Candidate-set coverage is 99.9996% for validation and 99.9979% for testing.

In [7]:
class UserHistory(Dataset):
    def __init__(self, matrix):
        self.matrix = matrix
    def __len__(self):
        return self.matrix.shape[0]
    def __getitem__(self, index):
        row = self.matrix.getrow(index).toarray().ravel().astype(np.float32)
        return torch.from_numpy(row)

train_data = UserHistory(user_item_matrix)

def make_loader():
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=device.type == "cuda", generator=generator)

first_batch = next(iter(make_loader()))
print("Batch oblik:", tuple(first_batch.shape))
print("Batch memorija:", round(first_batch.numel() * 4 / 1024 / 1024, 2), "MiB")

Batch oblik: (128, 10172)
Batch memorija: 4.97 MiB


The training matrix is sparse. The `UserHistory` class retrieves the required user row and converts it into a PyTorch vector. The `DataLoader` groups 128 user vectors into a batch of shape `(128, 10172)`, occupying approximately 4.97 MiB. This avoids storing dense vectors for all 700,111 users in RAM at once.

In [8]:
def eval_top_k(model, matrix, targets):
    model.eval()
    hit_sum = 0
    ndcg_sum = 0

    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = perf_counter()
    with torch.inference_mode():
        for start in range(0, len(targets), batch_size):
            batch = targets.iloc[start:start + batch_size]
            users = batch["user_idx"].to_numpy()
            target_items = torch.tensor(batch["item_idx"].to_numpy(), device=device)
            histories = matrix[users].toarray().astype(np.float32)
            histories = torch.from_numpy(histories).to(device)
            scores = model(histories)
            rec = torch.argsort(scores, dim=1, descending=True, stable=True)[:, :top_k]
            matches = rec.eq(target_items.unsqueeze(1))
            hits = matches.any(dim=1)
            ranks = matches.to(torch.int64).argmax(dim=1)
            hit_sum += hits.sum().item()
            ndcg_sum += (hits / torch.log2(ranks + 2)).sum().item()

    if device.type == "cuda":
        torch.cuda.synchronize()

    return {
        "hit_recall_at_10": hit_sum / len(targets),
        "ndcg_at_10": ndcg_sum / len(targets),
        "evaluation_seconds": perf_counter() - start_time
    }

The `eval_top_k` function follows the same protocol as the baselines. For each user, the model scores all 10,172 candidate apps, selects the top ten, and computes Hit/Recall@10 and NDCG@10. Both validation and test evaluation use only training interactions as user context, ensuring a consistent comparison across models.

## AutoRec Model

AutoRec encodes a binary user history into a compact hidden vector and decodes it into a score for each app. The encoder learns patterns of app co-occurrence in user histories, while the decoder restores the candidate-set dimensionality. In this study, the architecture is adapted to binary interactions rather than explicit ratings.

In [9]:
class AutoRec(nn.Module):
    def __init__(self, num_items, latent, dropout):
        super().__init__()
        self.encoder = nn.Linear(num_items, latent)
        self.dropout = nn.Dropout(dropout)
        self.decoder = nn.Linear(latent, num_items)
    def forward(self, history):
        hidden = torch.relu(self.encoder(history))
        hidden = self.dropout(hidden)
        return self.decoder(hidden)

positive = user_item_matrix.nnz
all_values = num_users * num_items
positive_weight = torch.tensor([(all_values - positive) / positive], device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=positive_weight)

print("Pozitivne interactions:", positive)
print("Positive weight:", round(positive_weight.item(), 2))

Pozitivne interactions: 17748311
Positive weight: 400.25


In [10]:
def train_ar_epoch(model, loader, optimizer):
    model.train()
    loss_sum = 0

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    start_time = perf_counter()

    for histories in loader:
        histories = histories.to(device)
        optimizer.zero_grad()
        scores = model(histories)
        loss = loss_fn(scores, histories)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(histories)

    peak_memory = None

    if device.type == "cuda":
        peak_memory = torch.cuda.max_memory_allocated() / 1024 / 1024

    return loss_sum / num_users, perf_counter() - start_time, peak_memory

def run_ar(config):
    set_seed(seed)
    model = AutoRec(num_items, config["latent"], config["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=0.00001)
    loader = make_loader()
    best_ndcg = -1
    best_epoch = 0
    no_change = 0
    best_state = None
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, epoch_time, peak_memory = train_ar_epoch(model, loader, optimizer)
        valid_eval = eval_top_k(model, user_item_matrix, valid_targets)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_hit_recall_10": valid_eval["hit_recall_at_10"],
            "validation_ndcg_10": valid_eval["ndcg_at_10"],
            "training_seconds": epoch_time,
            "peak_gpu_memory_mib": peak_memory })
        print(f"{config['name']} | epoch {epoch:02d} | loss: {train_loss:.4f} | NDCG@10: {valid_eval['ndcg_at_10']:.6f}")

        if valid_eval["ndcg_at_10"] > best_ndcg + min_improvement:
            best_ndcg = valid_eval["ndcg_at_10"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_change = 0
        else:
            no_change += 1

        if no_change >= patience:
            break

    history = pd.DataFrame(history)
    best_row = history.loc[history["epoch"] == best_epoch].iloc[0]

    result = {
        "name": config["name"],
        "latent": config["latent"],
        "dropout": config["dropout"],
        "lr": config["lr"],
        "best_epoch": best_epoch,
        "validation_hit_recall_10": best_row["validation_hit_recall_10"],
        "validation_ndcg_10": best_ndcg,
        "training_seconds": history["training_seconds"].sum(),
        "peak_gpu_memory_mib": history["peak_gpu_memory_mib"].max()}

    return best_state, history, result

AutoRec contains two linear layers. The encoder reduces the 10,172 input values to a hidden vector, followed by ReLU and dropout. The decoder then produces one score per app.

Observed interactions account for only a small fraction of possible pairs, so `BCEWithLogitsLoss` uses a `positive_weight` of approximately 400. This gives observed interactions greater influence and discourages the model from reducing loss merely by assigning low scores to the many unobserved entries. Adam updates the weights after each batch.

In [11]:
ar_options = [
    {"name": "AutoRec_1", "latent": 128, "dropout": 0.40, "lr": 0.001},
    {"name": "AutoRec_2", "latent": 256, "dropout": 0.40, "lr": 0.001},
    {"name": "AutoRec_3", "latent": 128, "dropout": 0.20, "lr": 0.001}
]

ar_rows = []
best_ar_state = None
best_ar_history = None
best_ar_result = None

for config in ar_options:
    state, history, result = run_ar(config)
    ar_rows.append(result)

    if best_ar_result is None or result["validation_ndcg_10"] > best_ar_result["validation_ndcg_10"]:
        best_ar_state = state
        best_ar_history = history
        best_ar_result = result

ar_tuning = pd.DataFrame(ar_rows)
ar_tuning = ar_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
ar_tuning.to_csv(results_dir / f"ar_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Izabrani AutoRec:", best_ar_result["name"])
ar_tuning.round(6)

AutoRec_1 | epoch 01 | loss: 0.7950 | NDCG@10: 0.001556
AutoRec_1 | epoch 02 | loss: 0.7195 | NDCG@10: 0.001385
AutoRec_1 | epoch 03 | loss: 0.7070 | NDCG@10: 0.001398
AutoRec_1 | epoch 04 | loss: 0.7017 | NDCG@10: 0.001455
AutoRec_2 | epoch 01 | loss: 0.7316 | NDCG@10: 0.001281
AutoRec_2 | epoch 02 | loss: 0.6299 | NDCG@10: 0.001300
AutoRec_2 | epoch 03 | loss: 0.6182 | NDCG@10: 0.001182
AutoRec_2 | epoch 04 | loss: 0.6132 | NDCG@10: 0.001259
AutoRec_2 | epoch 05 | loss: 0.6103 | NDCG@10: 0.001189
AutoRec_3 | epoch 01 | loss: 0.7581 | NDCG@10: 0.001259
AutoRec_3 | epoch 02 | loss: 0.6605 | NDCG@10: 0.001170
AutoRec_3 | epoch 03 | loss: 0.6442 | NDCG@10: 0.001071
AutoRec_3 | epoch 04 | loss: 0.6371 | NDCG@10: 0.001061
Izabrani AutoRec: AutoRec_1


,name,latent,dropout,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,AutoRec_1,128,0.4,0.001,1,0.003454,0.001556,529.471319,86.289062
1,AutoRec_2,256,0.4,0.001,2,0.002958,0.001300,705.122220,155.999512
2,AutoRec_3,128,0.2,0.001,1,0.002871,0.001259,531.779555,116.036621


In the first tuning round, `AutoRec_1` performs best with a hidden dimension of 128, dropout of 0.40, and a learning rate of 0.001. It achieves validation Hit/Recall@10 of 0.3454% and NDCG@10 of 0.1556%. A hidden dimension of 256 and lower dropout produce weaker results.

Validation performance declines within the first few epochs even as training loss decreases, suggesting possible overfitting. The best architecture is retained, and learning rates of 0.0005 and 0.002 are tested next.

In [12]:
ar_extra_options = [
    {"name": "AutoRec_4", "latent": best_ar_result["latent"],
     "dropout": best_ar_result["dropout"], "lr": 0.0005},
    {"name": "AutoRec_5", "latent": best_ar_result["latent"],
     "dropout": best_ar_result["dropout"], "lr": 0.002}
]

for config in ar_extra_options:
    state, history, result = run_ar(config)
    ar_rows.append(result)

    if result["validation_ndcg_10"] > best_ar_result["validation_ndcg_10"]:
        best_ar_state = state
        best_ar_history = history
        best_ar_result = result

ar_tuning = pd.DataFrame(ar_rows)
ar_tuning = ar_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
ar_tuning.to_csv(results_dir / f"ar_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Izabrani AutoRec nakon dodatne provere:", best_ar_result["name"])
ar_tuning.round(6)

AutoRec_4 | epoch 01 | loss: 0.8321 | NDCG@10: 0.002724
AutoRec_4 | epoch 02 | loss: 0.7456 | NDCG@10: 0.001648
AutoRec_4 | epoch 03 | loss: 0.7223 | NDCG@10: 0.001486
AutoRec_4 | epoch 04 | loss: 0.7111 | NDCG@10: 0.001357
AutoRec_5 | epoch 01 | loss: 0.7651 | NDCG@10: 0.001464
AutoRec_5 | epoch 02 | loss: 0.7146 | NDCG@10: 0.001484
AutoRec_5 | epoch 03 | loss: 0.7106 | NDCG@10: 0.001578
AutoRec_5 | epoch 04 | loss: 0.7086 | NDCG@10: 0.001606
AutoRec_5 | epoch 05 | loss: 0.7074 | NDCG@10: 0.001644
AutoRec_5 | epoch 06 | loss: 0.7067 | NDCG@10: 0.001653
AutoRec_5 | epoch 07 | loss: 0.7062 | NDCG@10: 0.001643
AutoRec_5 | epoch 08 | loss: 0.7058 | NDCG@10: 0.001524
Izabrani AutoRec nakon dodatne provere: AutoRec_4


,name,latent,dropout,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,AutoRec_4,128,0.4,0.0005,1,0.005649,0.002724,531.920886,106.102539
1,AutoRec_5,128,0.4,0.0020,5,0.003532,0.001644,1067.599987,96.129395
2,AutoRec_1,128,0.4,0.0010,1,0.003454,0.001556,529.471319,86.289062
3,AutoRec_2,256,0.4,0.0010,2,0.002958,0.001300,705.122220,155.999512
4,AutoRec_3,128,0.2,0.0010,1,0.002871,0.001259,531.779555,116.036621


Reducing the learning rate to 0.0005 improves performance. `AutoRec_4` achieves validation Hit/Recall@10 of 0.5649% and NDCG@10 of 0.2724%, while the higher learning rate of 0.002 performs worse.

Since the best result still occurs in the first epoch, a learning rate of 0.00025 and higher dropout of 0.50 are evaluated.

In [13]:
ar_last_options = [
    {"name": "AutoRec_6", "latent": 128, "dropout": 0.40, "lr": 0.00025},
    {"name": "AutoRec_7", "latent": 128, "dropout": 0.50, "lr": 0.0005}
]

for config in ar_last_options:
    state, history, result = run_ar(config)
    ar_rows.append(result)

    if result["validation_ndcg_10"] > best_ar_result["validation_ndcg_10"]:
        best_ar_state = state
        best_ar_history = history
        best_ar_result = result

ar_tuning = pd.DataFrame(ar_rows)
ar_tuning = ar_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
ar_tuning.to_csv(results_dir / f"ar_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani AutoRec:", best_ar_result["name"])
ar_tuning.round(6)

AutoRec_6 | epoch 01 | loss: 0.8798 | NDCG@10: 0.003855
AutoRec_6 | epoch 02 | loss: 0.7781 | NDCG@10: 0.002680
AutoRec_6 | epoch 03 | loss: 0.7498 | NDCG@10: 0.001978
AutoRec_6 | epoch 04 | loss: 0.7343 | NDCG@10: 0.001670
AutoRec_7 | epoch 01 | loss: 0.8452 | NDCG@10: 0.003234
AutoRec_7 | epoch 02 | loss: 0.7724 | NDCG@10: 0.002196
AutoRec_7 | epoch 03 | loss: 0.7524 | NDCG@10: 0.001924
AutoRec_7 | epoch 04 | loss: 0.7421 | NDCG@10: 0.001761
Konacno izabrani AutoRec: AutoRec_6


,name,latent,dropout,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,AutoRec_6,128,0.4,0.00025,1,0.008122,0.003855,536.803826,106.102539
1,AutoRec_7,128,0.5,0.00050,1,0.006729,0.003234,532.100655,96.129395
2,AutoRec_4,128,0.4,0.00050,1,0.005649,0.002724,531.920886,106.102539
3,AutoRec_5,128,0.4,0.00200,5,0.003532,0.001644,1067.599987,96.129395
4,AutoRec_1,128,0.4,0.00100,1,0.003454,0.001556,529.471319,86.289062
5,AutoRec_2,256,0.4,0.00100,2,0.002958,0.001300,705.122220,155.999512
6,AutoRec_3,128,0.2,0.00100,1,0.002871,0.001259,531.779555,116.036621


In the third round, `AutoRec_6`, with a learning rate of 0.00025, achieves NDCG@10 of 0.003855 and outperforms the configuration with dropout of 0.50. Its best result still occurs in the first epoch, motivating one further experiment with a learning rate of 0.0001.

In [14]:
ar_final_option = {
    "name": "AutoRec_8", "latent": 128,
    "dropout": 0.40, "lr": 0.0001
}

state, history, result = run_ar(ar_final_option)
ar_rows.append(result)

if result["validation_ndcg_10"] > best_ar_result["validation_ndcg_10"]:
    best_ar_state = state
    best_ar_history = history
    best_ar_result = result

ar_tuning = pd.DataFrame(ar_rows)
ar_tuning = ar_tuning.drop_duplicates(subset="name")
ar_tuning = ar_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
ar_tuning.to_csv(results_dir / f"ar_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani AutoRec:", best_ar_result["name"])
ar_tuning.round(6)

AutoRec_8 | epoch 01 | loss: 0.9874 | NDCG@10: 0.004480
AutoRec_8 | epoch 02 | loss: 0.8238 | NDCG@10: 0.004212
AutoRec_8 | epoch 03 | loss: 0.7962 | NDCG@10: 0.003619
AutoRec_8 | epoch 04 | loss: 0.7777 | NDCG@10: 0.003089
Konacno izabrani AutoRec: AutoRec_8


,name,latent,dropout,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,AutoRec_8,128,0.4,0.00010,1,0.009728,0.004480,533.888907,106.102539
1,AutoRec_6,128,0.4,0.00025,1,0.008122,0.003855,536.803826,106.102539
2,AutoRec_7,128,0.5,0.00050,1,0.006729,0.003234,532.100655,96.129395
3,AutoRec_4,128,0.4,0.00050,1,0.005649,0.002724,531.920886,106.102539
4,AutoRec_5,128,0.4,0.00200,5,0.003532,0.001644,1067.599987,96.129395
5,AutoRec_1,128,0.4,0.00100,1,0.003454,0.001556,529.471319,86.289062
6,AutoRec_2,256,0.4,0.00100,2,0.002958,0.001300,705.122220,155.999512
7,AutoRec_3,128,0.2,0.00100,1,0.002871,0.001259,531.779555,116.036621


In [15]:
ar_model = AutoRec(num_items, best_ar_result["latent"],
                   best_ar_result["dropout"]).to(device)
ar_model.load_state_dict(best_ar_state)
ar_test = eval_top_k(ar_model, user_item_matrix, test_targets)

ar_result = pd.DataFrame([{
    "model": "AutoRec",
    "users": num_users,
    "seed": seed,
    "validation_Hit@10": best_ar_result["validation_hit_recall_10"],
    "validation_Recall@10": best_ar_result["validation_hit_recall_10"],
    "validation_NDCG@10": best_ar_result["validation_ndcg_10"],
    "test_Hit@10": ar_test["hit_recall_at_10"],
    "test_Recall@10": ar_test["hit_recall_at_10"],
    "test_NDCG@10": ar_test["ndcg_at_10"],
    "test_coverage": test_coverage,
    "model_time_seconds": best_ar_result["training_seconds"],
    "notes": f"latent={best_ar_result['latent']}, dropout={best_ar_result['dropout']}, lr={best_ar_result['lr']}"
}])

best_ar_history.to_csv(results_dir / f"ar_history_{run_name}_seed_{seed}.csv", index=False)
torch.save({"state_dict": ar_model.state_dict(), "result": best_ar_result},
           results_dir / f"ar_{run_name}_seed_{seed}.pt")

print("AutoRec test:", ar_test)

AutoRec test: {'hit_recall_at_10': 0.010557123594478472, 'ndcg_at_10': 0.004553892390508096, 'evaluation_seconds': 72.37403870001435}


The selected configuration is `AutoRec_8`, with a hidden dimension of 128, dropout of 0.40, and a learning rate of 0.0001. Its validation Hit/Recall@10 is 0.9728%, and NDCG@10 is 0.4480%.

On the test set, it achieves Hit/Recall@10 of 1.0557% and NDCG@10 of 0.4554%. Only training histories are used as test-time input, and the hyperparameters and checkpoint are selected without using test results.

## MultiVAE Model

MultiVAE is a variational autoencoder designed for implicit-feedback recommendation. It learns a distribution over latent representations from user histories, and its decoder maps a latent vector to app scores. Compared with AutoRec, it introduces a probabilistic representation and a KL-divergence loss term while retaining the same interaction matrix, split, and evaluation protocol.

In [16]:
class MultiVAE(nn.Module):
    def __init__(self, num_items, hidden, latent, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.Linear(num_items, hidden)
        self.mean_logvar = nn.Linear(hidden, latent * 2)
        self.decoder = nn.Linear(latent, hidden)
        self.output = nn.Linear(hidden, num_items)

    def forward(self, history, values=False):
        history = F.normalize(history, p=2, dim=1)
        hidden = torch.tanh(self.encoder(self.dropout(history)))
        mean, logvar = torch.chunk(self.mean_logvar(hidden), 2, dim=1)

        if self.training:
            latent = mean + torch.randn_like(mean) * torch.exp(0.5 * logvar)
        else:
            latent = mean
        scores = self.output(torch.tanh(self.decoder(latent)))
        if values:
            return scores, mean, logvar
        return scores

def multivae_loss(scores, history, mean, logvar, kl_weight):
    reconstruction = -(F.log_softmax(scores, dim=1) * history).sum(dim=1).mean()
    kl = -0.5 * (1 + logvar - mean.pow(2) - logvar.exp()).sum(dim=1).mean()
    return reconstruction + kl_weight * kl, reconstruction, kl

User histories are normalized before entering the encoder. During training, input dropout randomly masks part of each history. The encoder produces a hidden representation followed by the mean and log variance of the latent distribution. Latent vectors are sampled during training; evaluation uses the mean for deterministic predictions.

The first tuning round varies the latent dimension, dropout, and maximum KL weight while keeping the learning rate fixed.

In [17]:
kl_steps = 2000

def train_multivae_epoch(model, loader, optimizer, step, max_kl):
    model.train()
    loss_sum = 0
    reconstruction_sum = 0
    kl_sum = 0

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    start_time = perf_counter()

    for histories in loader:
        histories = histories.to(device)
        kl_weight = min(step / kl_steps, 1) * max_kl
        optimizer.zero_grad()
        scores, mean, logvar = model(histories, values=True)
        loss, reconstruction, kl = multivae_loss(scores, histories, mean, logvar, kl_weight)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * len(histories)
        reconstruction_sum += reconstruction.item() * len(histories)
        kl_sum += kl.item() * len(histories)
        step += 1

    peak_memory = None

    if device.type == "cuda":
        peak_memory = torch.cuda.max_memory_allocated() / 1024 / 1024

    return {
        "loss": loss_sum / num_users,
        "reconstruction": reconstruction_sum / num_users,
        "kl": kl_sum / num_users,
        "time": perf_counter() - start_time,
        "memory": peak_memory
    }, step

def run_multivae(config):
    set_seed(seed)
    model = MultiVAE(num_items, config["hidden"], config["latent"], config["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loader = make_loader()
    best_ndcg = -1
    best_epoch = 0
    no_change = 0
    best_state = None
    history = []
    step = 0

    for epoch in range(1, max_epochs + 1):
        train, step = train_multivae_epoch(model, loader, optimizer, step, config["max_kl"])
        valid_eval = eval_top_k(model, user_item_matrix, valid_targets)

        history.append({
            "epoch": epoch,
            "train_loss": train["loss"],
            "reconstruction_loss": train["reconstruction"],
            "kl_loss": train["kl"],
            "validation_hit_recall_10": valid_eval["hit_recall_at_10"],
            "validation_ndcg_10": valid_eval["ndcg_at_10"],
            "training_seconds": train["time"],
            "peak_gpu_memory_mib": train["memory"]
        })

        print(f"{config['name']} | epoch {epoch:02d} | loss: {train['loss']:.4f} | NDCG@10: {valid_eval['ndcg_at_10']:.6f}")

        if valid_eval["ndcg_at_10"] > best_ndcg + min_improvement:
            best_ndcg = valid_eval["ndcg_at_10"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_change = 0
        else:
            no_change += 1

        if no_change >= patience:
            break

    history = pd.DataFrame(history)
    best_row = history.loc[history["epoch"] == best_epoch].iloc[0]

    result = {
        "name": config["name"],
        "hidden": config["hidden"],
        "latent": config["latent"],
        "dropout": config["dropout"],
        "max_kl": config["max_kl"],
        "lr": config["lr"],
        "best_epoch": best_epoch,
        "validation_hit_recall_10": best_row["validation_hit_recall_10"],
        "validation_ndcg_10": best_ndcg,
        "training_seconds": history["training_seconds"].sum(),
        "peak_gpu_memory_mib": history["peak_gpu_memory_mib"].max()
    }

    return best_state, history, result

The loss combines a reconstruction term and KL divergence. The reconstruction term rewards scores assigned to observed apps, while KL divergence regularizes the latent space. Its weight increases gradually over the first 2,000 batches to avoid excessive regularization at the start of training.

Validation NDCG@10 is computed after every epoch, and the best checkpoint is retained using the same early-stopping rule.

In [18]:
multivae_options = [
    {"name": "MultiVAE_1", "hidden": 256, "latent": 128, "dropout": 0.50, "max_kl": 0.20, "lr": 0.001},
    {"name": "MultiVAE_2", "hidden": 256, "latent": 128, "dropout": 0.30, "max_kl": 0.20, "lr": 0.001},
    {"name": "MultiVAE_3", "hidden": 256, "latent": 64, "dropout": 0.50, "max_kl": 0.20, "lr": 0.001},
    {"name": "MultiVAE_4", "hidden": 256, "latent": 128, "dropout": 0.50, "max_kl": 0.10, "lr": 0.001}
]

multivae_rows = []
best_multivae_state = None
best_multivae_history = None
best_multivae_result = None

for config in multivae_options:
    state, history, result = run_multivae(config)
    multivae_rows.append(result)

    if best_multivae_result is None or result["validation_ndcg_10"] > best_multivae_result["validation_ndcg_10"]:
        best_multivae_state = state
        best_multivae_history = history
        best_multivae_result = result

multivae_tuning = pd.DataFrame(multivae_rows)
multivae_tuning = multivae_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
multivae_tuning.to_csv(results_dir / f"multivae_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Izabrani MultiVAE:", best_multivae_result["name"])
multivae_tuning.round(6)

MultiVAE_1 | epoch 01 | loss: 223.4078 | NDCG@10: 0.003032
MultiVAE_1 | epoch 02 | loss: 221.7630 | NDCG@10: 0.001898
MultiVAE_1 | epoch 03 | loss: 220.7883 | NDCG@10: 0.001668
MultiVAE_1 | epoch 04 | loss: 220.2011 | NDCG@10: 0.001648
MultiVAE_2 | epoch 01 | loss: 221.6461 | NDCG@10: 0.001312
MultiVAE_2 | epoch 02 | loss: 217.9966 | NDCG@10: 0.001070
MultiVAE_2 | epoch 03 | loss: 216.2808 | NDCG@10: 0.000957
MultiVAE_2 | epoch 04 | loss: 215.2271 | NDCG@10: 0.000951
MultiVAE_3 | epoch 01 | loss: 223.1952 | NDCG@10: 0.002862
MultiVAE_3 | epoch 02 | loss: 221.8947 | NDCG@10: 0.002254
MultiVAE_3 | epoch 03 | loss: 221.3328 | NDCG@10: 0.001952
MultiVAE_3 | epoch 04 | loss: 221.0226 | NDCG@10: 0.002002
MultiVAE_4 | epoch 01 | loss: 221.3861 | NDCG@10: 0.001557
MultiVAE_4 | epoch 02 | loss: 217.6562 | NDCG@10: 0.001280
MultiVAE_4 | epoch 03 | loss: 216.0501 | NDCG@10: 0.001173
MultiVAE_4 | epoch 04 | loss: 215.0926 | NDCG@10: 0.001193
Izabrani MultiVAE: MultiVAE_1


,name,hidden,latent,dropout,max_kl,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,MultiVAE_1,256,128,0.5,0.2,0.001,1,0.006515,0.003032,582.422098,168.360352
1,MultiVAE_3,256,64,0.5,0.2,0.001,1,0.006176,0.002862,579.696896,207.738281
2,MultiVAE_4,256,128,0.5,0.1,0.001,1,0.003602,0.001557,585.642468,208.607910
3,MultiVAE_2,256,128,0.3,0.2,0.001,1,0.003021,0.001312,582.366417,188.644531


In the first round, `MultiVAE_1` performs best with a hidden dimension of 256, latent dimension of 128, dropout of 0.50, and maximum KL weight of 0.20. It achieves validation Hit/Recall@10 of 0.6515% and NDCG@10 of 0.3032%.

The next round retains this architecture and tests a lower learning rate of 0.0005. A larger hidden layer with 512 units is also evaluated separately.

In [19]:
multivae_extra_options = [
    {"name": "MultiVAE_5", "hidden": best_multivae_result["hidden"],
     "latent": best_multivae_result["latent"], "dropout": best_multivae_result["dropout"],
     "max_kl": best_multivae_result["max_kl"], "lr": 0.0005},
    {"name": "MultiVAE_6", "hidden": 512,
     "latent": best_multivae_result["latent"], "dropout": best_multivae_result["dropout"],
     "max_kl": best_multivae_result["max_kl"], "lr": best_multivae_result["lr"]}
]

for config in multivae_extra_options:
    state, history, result = run_multivae(config)
    multivae_rows.append(result)

    if result["validation_ndcg_10"] > best_multivae_result["validation_ndcg_10"]:
        best_multivae_state = state
        best_multivae_history = history
        best_multivae_result = result

multivae_tuning = pd.DataFrame(multivae_rows)
multivae_tuning = multivae_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
multivae_tuning.to_csv(results_dir / f"multivae_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Izabrani MultiVAE nakon dodatne provere:", best_multivae_result["name"])
multivae_tuning.round(6)

MultiVAE_5 | epoch 01 | loss: 223.7603 | NDCG@10: 0.005163
MultiVAE_5 | epoch 02 | loss: 222.6322 | NDCG@10: 0.002818
MultiVAE_5 | epoch 03 | loss: 221.4582 | NDCG@10: 0.002037
MultiVAE_5 | epoch 04 | loss: 220.6498 | NDCG@10: 0.001927
MultiVAE_6 | epoch 01 | loss: 222.9554 | NDCG@10: 0.002103
MultiVAE_6 | epoch 02 | loss: 220.9751 | NDCG@10: 0.001675
MultiVAE_6 | epoch 03 | loss: 220.0319 | NDCG@10: 0.001579
MultiVAE_6 | epoch 04 | loss: 219.4991 | NDCG@10: 0.001600
Izabrani MultiVAE nakon dodatne provere: MultiVAE_5


,name,hidden,latent,dropout,max_kl,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,MultiVAE_5,256,128,0.5,0.2,0.0005,1,0.010518,0.005163,587.963125,208.795898
1,MultiVAE_1,256,128,0.5,0.2,0.0010,1,0.006515,0.003032,582.422098,168.360352
2,MultiVAE_3,256,64,0.5,0.2,0.0010,1,0.006176,0.002862,579.696896,207.738281
3,MultiVAE_6,512,128,0.5,0.2,0.0010,1,0.004761,0.002103,638.284685,311.437500
4,MultiVAE_4,256,128,0.5,0.1,0.0010,1,0.003602,0.001557,585.642468,208.607910
5,MultiVAE_2,256,128,0.3,0.2,0.0010,1,0.003021,0.001312,582.366417,188.644531


With a learning rate of 0.0005, `MultiVAE_5` improves validation NDCG@10 to 0.005163. The larger hidden layer with 512 units yields a lower score of 0.002103, so model capacity is not increased further.

The next experiments test a learning rate of 0.00025 and a maximum KL weight of 0.30, retaining the hidden dimension of 256 and latent dimension of 128.

In [20]:
multivae_last_options = [
    {"name": "MultiVAE_7", "hidden": 256, "latent": 128,
     "dropout": 0.50, "max_kl": 0.20, "lr": 0.00025},
    {"name": "MultiVAE_8", "hidden": 256, "latent": 128,
     "dropout": 0.50, "max_kl": 0.30, "lr": 0.0005}
]

for config in multivae_last_options:
    state, history, result = run_multivae(config)
    multivae_rows.append(result)

    if result["validation_ndcg_10"] > best_multivae_result["validation_ndcg_10"]:
        best_multivae_state = state
        best_multivae_history = history
        best_multivae_result = result

multivae_tuning = pd.DataFrame(multivae_rows)
multivae_tuning = multivae_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
multivae_tuning.to_csv(results_dir / f"multivae_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani MultiVAE:", best_multivae_result["name"])
multivae_tuning.round(6)

MultiVAE_7 | epoch 01 | loss: 224.1063 | NDCG@10: 0.005741
MultiVAE_7 | epoch 02 | loss: 223.2839 | NDCG@10: 0.004846
MultiVAE_7 | epoch 03 | loss: 222.8113 | NDCG@10: 0.003872
MultiVAE_7 | epoch 04 | loss: 222.1797 | NDCG@10: 0.003005
MultiVAE_8 | epoch 01 | loss: 223.9204 | NDCG@10: 0.005864
MultiVAE_8 | epoch 02 | loss: 223.5767 | NDCG@10: 0.005457
MultiVAE_8 | epoch 03 | loss: 223.5368 | NDCG@10: 0.005426
MultiVAE_8 | epoch 04 | loss: 223.4472 | NDCG@10: 0.005292
Konacno izabrani MultiVAE: MultiVAE_8


,name,hidden,latent,dropout,max_kl,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,MultiVAE_8,256,128,0.5,0.3,0.00050,1,0.011960,0.005864,583.241000,188.378906
1,MultiVAE_7,256,128,0.5,0.2,0.00025,1,0.011630,0.005741,583.564451,229.172852
2,MultiVAE_5,256,128,0.5,0.2,0.00050,1,0.010518,0.005163,587.963125,208.795898
3,MultiVAE_1,256,128,0.5,0.2,0.00100,1,0.006515,0.003032,582.422098,168.360352
4,MultiVAE_3,256,64,0.5,0.2,0.00100,1,0.006176,0.002862,579.696896,207.738281
5,MultiVAE_6,512,128,0.5,0.2,0.00100,1,0.004761,0.002103,638.284685,311.437500
6,MultiVAE_4,256,128,0.5,0.1,0.00100,1,0.003602,0.001557,585.642468,208.607910
7,MultiVAE_2,256,128,0.3,0.2,0.00100,1,0.003021,0.001312,582.366417,188.644531


`MultiVAE_8`, with a learning rate of 0.0005 and maximum KL weight of 0.30, achieves NDCG@10 of 0.005864. `MultiVAE_7`, with a learning rate of 0.00025 and KL weight of 0.20, achieves 0.005741. Since these experiments modify different hyperparameters, the final configuration combines the lower learning rate of 0.00025 with the KL weight of 0.30.

In [21]:
config = {"name": "MultiVAE_9", "hidden": 256, "latent": 128,
          "dropout": 0.50, "max_kl": 0.30, "lr": 0.00025}

state, history, result = run_multivae(config)
multivae_rows.append(result)

if result["validation_ndcg_10"] > best_multivae_result["validation_ndcg_10"]:
    best_multivae_state = state
    best_multivae_history = history
    best_multivae_result = result

multivae_tuning = pd.DataFrame(multivae_rows)
multivae_tuning = multivae_tuning.drop_duplicates(subset="name")
multivae_tuning = multivae_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
multivae_tuning.to_csv(results_dir / f"multivae_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani MultiVAE:", best_multivae_result["name"])
multivae_tuning.round(6)

MultiVAE_9 | epoch 01 | loss: 224.1644 | NDCG@10: 0.005918
MultiVAE_9 | epoch 02 | loss: 223.5298 | NDCG@10: 0.005436
MultiVAE_9 | epoch 03 | loss: 223.4883 | NDCG@10: 0.005474
MultiVAE_9 | epoch 04 | loss: 223.4544 | NDCG@10: 0.005909
Konacno izabrani MultiVAE: MultiVAE_9


,name,hidden,latent,dropout,max_kl,lr,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,MultiVAE_9,256,128,0.5,0.3,0.00025,1,0.012002,0.005918,583.880002,188.378906
1,MultiVAE_8,256,128,0.5,0.3,0.00050,1,0.011960,0.005864,583.241000,188.378906
2,MultiVAE_7,256,128,0.5,0.2,0.00025,1,0.011630,0.005741,583.564451,229.172852
3,MultiVAE_5,256,128,0.5,0.2,0.00050,1,0.010518,0.005163,587.963125,208.795898
4,MultiVAE_1,256,128,0.5,0.2,0.00100,1,0.006515,0.003032,582.422098,168.360352
5,MultiVAE_3,256,64,0.5,0.2,0.00100,1,0.006176,0.002862,579.696896,207.738281
6,MultiVAE_6,512,128,0.5,0.2,0.00100,1,0.004761,0.002103,638.284685,311.437500
7,MultiVAE_4,256,128,0.5,0.1,0.00100,1,0.003602,0.001557,585.642468,208.607910
8,MultiVAE_2,256,128,0.3,0.2,0.00100,1,0.003021,0.001312,582.366417,188.644531


The selected configuration is `MultiVAE_9`, with a hidden dimension of 256, latent dimension of 128, dropout of 0.50, maximum KL weight of 0.30, and a learning rate of 0.00025. Its best validation result occurs in the first epoch: Hit/Recall@10 is 1.2002%, and NDCG@10 is 0.5918%. The fourth-epoch result is very close but does not exceed the first.

In [22]:
multivae_model = MultiVAE(num_items, best_multivae_result["hidden"],
                          best_multivae_result["latent"],
                          best_multivae_result["dropout"]).to(device)
multivae_model.load_state_dict(best_multivae_state)
multivae_test = eval_top_k(multivae_model, user_item_matrix, test_targets)

multivae_result = pd.DataFrame([{
    "model": "MultiVAE",
    "users": num_users,
    "seed": seed,
    "validation_Hit@10": best_multivae_result["validation_hit_recall_10"],
    "validation_Recall@10": best_multivae_result["validation_hit_recall_10"],
    "validation_NDCG@10": best_multivae_result["validation_ndcg_10"],
    "test_Hit@10": multivae_test["hit_recall_at_10"],
    "test_Recall@10": multivae_test["hit_recall_at_10"],
    "test_NDCG@10": multivae_test["ndcg_at_10"],
    "test_coverage": test_coverage,
    "model_time_seconds": best_multivae_result["training_seconds"],
    "notes": f"hidden={best_multivae_result['hidden']}, latent={best_multivae_result['latent']}, dropout={best_multivae_result['dropout']}, max_kl={best_multivae_result['max_kl']}"
}])

best_multivae_history.to_csv(results_dir / f"multivae_history_{run_name}_seed_{seed}.csv", index=False)
torch.save({"state_dict": multivae_model.state_dict(), "result": best_multivae_result},
           results_dir / f"multivae_{run_name}_seed_{seed}.pt")

print("MultiVAE test:", multivae_test)

MultiVAE test: {'hit_recall_at_10': 0.013859527836182467, 'ndcg_at_10': 0.006911976276165986, 'evaluation_seconds': 67.0608726000064}


`MultiVAE_9` is evaluated once on the test set. It achieves Hit/Recall@10 of 1.3860% and NDCG@10 of 0.6912%, with evaluation taking approximately 81 seconds. As with AutoRec, only training histories are used as test-time input.

## Comparison with the Baselines

In [23]:
baseline_results = pd.read_csv(baseline_path)
neural_results = pd.concat([ar_result, multivae_result], ignore_index=True)
all_results = pd.concat([baseline_results, neural_results], ignore_index=True)
all_results = all_results.sort_values("test_NDCG@10", ascending=False).reset_index(drop=True)

all_results.to_csv(results_dir / f"all_results_{run_name}_seed_{seed}.csv", index=False)

show_cols = [
    "model", "validation_Hit@10", "validation_NDCG@10",
    "test_Hit@10", "test_NDCG@10", "model_time_seconds", "notes"
]

all_results[show_cols].round(6)

,model,validation_Hit@10,validation_NDCG@10,test_Hit@10,test_NDCG@10,model_time_seconds,notes
0,ItemKNN,0.013441,0.006680,0.016872,0.008256,36.727301,Binarni ItemKNN sa k = 200
1,Popularity,0.012481,0.006411,0.015051,0.007762,0.227138,Globalna popularnost iz train skupa
2,MultiVAE,0.012002,0.005918,0.013860,0.006912,583.880002,"hidden=256, latent=128, dropout=0.5, max_kl=0.3"
3,AutoRec,0.009728,0.004480,0.010557,0.004554,533.888907,"latent=128, dropout=0.4, lr=0.0001"


ItemKNN achieves the highest test NDCG@10 at 0.8256%, followed by Popularity at 0.7762%. MultiVAE achieves 0.6912%, and AutoRec achieves 0.4554%.

MultiVAE performs better than AutoRec in these experiments, suggesting that its probabilistic latent representation is better suited to this implicit-interaction setting. Neither autoencoder outperforms the baselines. All models exceed the uniform-random expectation of approximately 0.0983% for Hit@10 and 0.0447% for NDCG@10.